# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb

HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

Paste your Hugging Face READ token: ··········


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis + Time Window

Unit of analysis:
One row represents the daily search performance of one content item (content_hash_id) for one client (client_hash_id) on one report_date.

Time window:
This analysis uses data from March 2026 (month = '2026-03') because it is a mid-panel month and avoids using the final month as recommended in the assignment.

Prediction goal:
The goal is to identify content that is likely to decline in search performance so it can be prioritized for refresh.

Output:
The final output will be a ranked list of content pages that should be considered for refresh.

In [6]:
con.sql(f"""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Feature

- gsc_impressions
  Knowable before prediction and measures search visibility.

- gsc_clicks
  Measures user engagement from search.

- gsc_avg_position
  Represents average ranking position.

- visible_queries
  Indicates how many search queries lead to the content.

- top_query_share
  Measures dependence on a single search query.

---

## Label

- is_declining

Proxy label indicating whether impressions declined by more than 20% compared to the previous 30-day period.

This is the prediction target and must never be used as a feature.

---

## Context

- client_hash_id
- content_hash_id
- report_date

Used for grouping, joining, filtering, and splitting data.

These identifiers are never used as model features.

---

## Excluded

trend_pct
Reason:
Derived from future performance and causes label leakage.

trend_direction
Reason:
Computed directly from trend_pct.

Revenue columns (if available)
Reason:
Not available at prediction time.

Private client identifiers
Reason:
Contain identifying information rather than predictive value.

In [7]:
con.sql(f"""
SELECT
AVG(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_impressions,
AVG(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_clicks,
AVG(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS missing_position
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,missing_impressions,missing_clicks,missing_position
0,0.0,0.0,0.633074


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 – Verify the Grain

This query checks whether one row truly represents one content item for one client on one day.

If no rows are returned, the grain definition is correct.

In [8]:
con.sql(f"""
SELECT
client_hash_id,
content_hash_id,
report_date,
COUNT(*) AS duplicates
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY 1,2,3
HAVING COUNT(*)>1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,duplicates


### Query 2 – Verify the Time Window

This query confirms the number of rows and the minimum and maximum report dates for the selected month.

In [9]:
con.sql(f"""
SELECT
COUNT(*) AS rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

,rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


### Query 3 – Verify Data Availability

This query checks how many rows contain usable GA4 data.

In [10]:
con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE ga4_data_available IS TRUE
AND month='2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

1. The dataset cannot prove that refreshing content caused performance improvements because it only measures observed behavior.

2. Different clients have different history lengths, so comparisons across clients should be interpreted carefully.

3. GA4 metrics are unavailable before each client's ga4_data_start date.

4. Search behavior changes over time, so relationships observed during March 2026 may not generalize to future periods.

5. This dataset supports decision-making but should not be interpreted as evidence of causation.

In [11]:
con.sql(f"""
SELECT
MIN(gsc_data_start) AS earliest_start,
MAX(gsc_data_start) AS latest_start
FROM {TABLES['dim_clients']}
""").df()

,earliest_start,latest_start
0,2025-01-27,2026-06-02


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.